In [2]:
import numpy as np
import sqlalchemy as db
import pandas as pd
import talib as ta

In [3]:
engine = db.create_engine('sqlite:///../raw/data.db') #loads db from raw

In [4]:
#loads all tables from db splitting events if they have data or don't
data_stocks = pd.read_sql('SELECT * FROM Stocks', con=engine) #Loads Stocks table
data_fed = pd.read_sql('SELECT * FROM Macro', con=engine) #loads macro data table
data_events = pd.read_sql('SELECT * FROM Events WHERE actual != "None" OR previous != "None"', con=engine) #load events with data
events_no_data = pd.read_sql('SELECT * FROM Events WHERE actual = "None" AND previous = "None"', con=engine) #load events without data

In [5]:
data_stocks.head()

,date,ticker,open,high,low,close,volume
0,1993-01-29 00:00:00.000000,SPY,24.258655,24.258655,24.137965,24.241413,1003200
1,1993-02-01 00:00:00.000000,SPY,24.258648,24.413820,24.258648,24.413820,480500
2,1993-02-02 00:00:00.000000,SPY,24.396588,24.482795,24.344863,24.465553,201300
3,1993-02-03 00:00:00.000000,SPY,24.500034,24.741414,24.482793,24.724173,529400
4,1993-02-04 00:00:00.000000,SPY,24.810374,24.879340,24.534512,24.827616,531500


In [6]:
data_stocks.info()

<class 'pandas.DataFrame'>
RangeIndex: 44603 entries, 0 to 44602
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    44603 non-null  str    
 1   ticker  44603 non-null  str    
 2   open    44603 non-null  float64
 3   high    44603 non-null  float64
 4   low     44603 non-null  float64
 5   close   44603 non-null  float64
 6   volume  44603 non-null  int64  
dtypes: float64(4), int64(1), str(2)
memory usage: 3.7 MB


In [7]:
data_stocks['ticker'].unique()

<ArrowStringArray>
['SPY', 'QQQ', '^VIX', 'DX-Y.NYB', 'GC=F']
Length: 5, dtype: str

##### Creates the technical analisis here for each stock

In [26]:
def technical_analysis(df):
    close = df['close'].astype(float)
    high = df['high'].astype(float)
    low = df['low'].astype(float)
    volume = df['volume'].astype(float)

    # --- Overlap Studies ---
    df['EMA_14'] = ta.EMA(close, timeperiod=14)
    df['EMA_30'] = ta.EMA(close, timeperiod=30)
    df['EMA_60'] = ta.EMA(close, timeperiod=60)

    df['SMA_14'] = ta.SMA(close, timeperiod=14)
    df['SMA_30'] = ta.SMA(close, timeperiod=30)
    df['SMA_60'] = ta.SMA(close, timeperiod=60)

    df['BB_upper'], df['BB_middle'], df['BB_lower'] = ta.BBANDS(close, timeperiod=20)

    # --- Momentum Indicators ---
    df['RSI_14'] = ta.RSI(close, timeperiod=14)
    df['RSI_21'] = ta.RSI(close, timeperiod=21)
    df['RSI_30'] = ta.RSI(close, timeperiod=30)


    df['MACD'], df['MACD_signal'], df['MACD_hist'] = ta.MACD(close, fastperiod=12, slowperiod=26, signalperiod=9)

    df['AROON_down'], df['AROON_up'] = ta.AROON(high, low, timeperiod=14)

    df['MFI'] = ta.MFI(high, low, close, volume, timeperiod=14)

    # --- Price Transform ---
    df['AVGPRICE'] = ta.AVGPRICE(df['open'], high, low, close)
    df['MEDPRICE'] = ta.MEDPRICE(high, low)

    df = df.dropna()

    return df


In [42]:
df_sp500 = technical_analysis(data_stocks[data_stocks['ticker'] == 'SPY'])
df_qqq = technical_analysis(data_stocks[data_stocks['ticker'] == 'QQQ'])
df_vix = technical_analysis(data_stocks[data_stocks['ticker'] == '^VIX'])
df_dxy = technical_analysis(data_stocks[data_stocks['ticker'] == 'DX-Y.NYB'])
df_gold = technical_analysis(data_stocks[data_stocks['ticker'] == 'GC=F'])

In [34]:
df_dxy.head()

,date,ticker,open,high,low,close,volume,EMA_14,EMA_30,EMA_60,...,RSI_21,RSI_30,MACD,MACD_signal,MACD_hist,AROON_down,AROON_up,MFI,AVGPRICE,MEDPRICE
24265,1971-03-30 00:00:00.000000,DX-Y.NYB,120.180000,120.180000,120.180000,120.180000,0,120.175946,120.200062,120.266333,...,41.390805,37.903522,-0.016998,-0.024081,0.007082,0.000000,28.571429,0.0,120.180000,120.180000
24266,1971-03-31 00:00:00.000000,DX-Y.NYB,120.169998,120.169998,120.169998,120.169998,0,120.175153,120.198122,120.263175,...,40.158540,37.183568,-0.016136,-0.022492,0.006356,64.285714,21.428571,0.0,120.169998,120.169998
24267,1971-04-01 00:00:00.000000,DX-Y.NYB,120.180000,120.180000,120.180000,120.180000,0,120.175799,120.196953,120.260448,...,41.972483,38.394086,-0.014478,-0.020889,0.006411,57.142857,14.285714,0.0,120.180000,120.180000
24268,1971-04-02 00:00:00.000000,DX-Y.NYB,120.180000,120.180000,120.180000,120.180000,0,120.176359,120.195859,120.257810,...,41.972483,38.394086,-0.013015,-0.019314,0.006300,50.000000,7.142857,0.0,120.180000,120.180000
24269,1971-04-05 00:00:00.000000,DX-Y.NYB,120.190002,120.190002,120.190002,120.190002,0,120.178178,120.195481,120.255587,...,43.849021,39.638892,-0.010922,-0.017636,0.006714,42.857143,100.000000,0.0,120.190002,120.190002


In [47]:
def col_names(df):

    ticker_name = str(df['ticker'].iloc[0]).lower()

    df = df.drop(columns=['ticker'])

    df.columns = [f'{ticker_name}_{col}' for col in df.columns]

    return df

df_sp500_format = col_names(df_sp500)
df_qqq_format = col_names(df_qqq)
df_vix_format = col_names(df_vix)
df_dxy_format = col_names(df_dxy)
df_gold_format = col_names(df_gold)

df_stocks = pd.concat([df_sp500_format, df_qqq_format, df_vix_format, df_dxy_format, df_gold_format], axis=1)

df_stocks.head()

,spy_date,spy_open,spy_high,spy_low,spy_close,spy_volume,spy_EMA_14,spy_EMA_30,spy_EMA_60,spy_SMA_14,...,gc=f_RSI_21,gc=f_RSI_30,gc=f_MACD,gc=f_MACD_signal,gc=f_MACD_hist,gc=f_AROON_down,gc=f_AROON_up,gc=f_MFI,gc=f_AVGPRICE,gc=f_MEDPRICE


## Transform one colum for each fed series
#### Saved parquet

In [10]:
data_fed.head()

,date,serie,value
0,1954-07-01 00:00:00.000000,FEDFUNDS,0.80
1,1954-08-01 00:00:00.000000,FEDFUNDS,1.22
2,1954-09-01 00:00:00.000000,FEDFUNDS,1.07
3,1954-10-01 00:00:00.000000,FEDFUNDS,0.85
4,1954-11-01 00:00:00.000000,FEDFUNDS,0.83


In [11]:
data_fed.info()

<class 'pandas.DataFrame'>
RangeIndex: 99434 entries, 0 to 99433
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    99434 non-null  str    
 1   serie   99434 non-null  str    
 2   value   99434 non-null  float64
dtypes: float64(1), str(2)
memory usage: 5.3 MB


In [43]:
cols = data_fed['serie'].unique()
cols

<ArrowStringArray>
[    'FEDFUNDS',          'DFF',       'T10Y2Y',       'T10Y3M',
         'GS10',          'GS2', 'BAMLH0A0HYM2',   'BAMLC0A0CM',
         'NFCI',       'UNRATE',     'CPIAUCSL',        'T5YIE',
       'T10YIE',         'ICSA',        'WALCL',         'M2SL',
         'SOFR',      'TEDRATE']
Length: 18, dtype: str

In [49]:
#get all dates from original dataframe
data_fed['date'] = pd.to_datetime(data_fed['date'])
#sort all unique dates
all_dates = sorted(data_fed['date'].unique(), reverse=False)
#create the new dataframe and set all sorted dates as index
data_fed_transposed = pd.DataFrame(index=all_dates)

#creates one new colum for each serie and assign all the values into the correct date
for colum in cols:
    if colum == 'SOFR':
        pass
    else:
        series_data = data_fed[data_fed['serie'] == colum].set_index('date')['value']
        data_fed_transposed[colum] = series_data

In [50]:
data_fed_transposed.tail()

,FEDFUNDS,DFF,T10Y2Y,T10Y3M,GS10,GS2,BAMLH0A0HYM2,BAMLC0A0CM,NFCI,UNRATE,CPIAUCSL,T5YIE,T10YIE,ICSA,WALCL,M2SL,TEDRATE
2026-02-18,NaN,3.64,0.62,0.39,NaN,NaN,2.86,0.78,NaN,NaN,NaN,2.43,2.29,NaN,6622382.0,NaN,NaN
2026-02-19,NaN,3.64,0.61,0.39,NaN,NaN,2.88,0.79,NaN,NaN,NaN,2.43,2.29,206000.0,NaN,NaN,NaN
2026-02-20,NaN,3.64,0.60,0.39,NaN,NaN,2.86,0.78,-0.56814,NaN,NaN,2.43,2.28,NaN,NaN,NaN,NaN
2026-02-23,NaN,NaN,0.60,0.34,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.40,2.26,NaN,NaN,NaN,NaN
2026-02-25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6613395.0,NaN,NaN


In [51]:
#now we need to fill al Nans with the previous values
data_fed_transposed = data_fed_transposed.ffill(axis=0) #ffill gets the last valid value and fills the next nans
data_fed_transposed.tail()

,FEDFUNDS,DFF,T10Y2Y,T10Y3M,GS10,GS2,BAMLH0A0HYM2,BAMLC0A0CM,NFCI,UNRATE,CPIAUCSL,T5YIE,T10YIE,ICSA,WALCL,M2SL,TEDRATE
2026-02-18,3.64,3.64,0.62,0.39,4.21,3.54,2.86,0.78,-0.56857,4.3,326.588,2.43,2.29,229000.0,6622382.0,22411.0,0.09
2026-02-19,3.64,3.64,0.61,0.39,4.21,3.54,2.88,0.79,-0.56857,4.3,326.588,2.43,2.29,206000.0,6622382.0,22411.0,0.09
2026-02-20,3.64,3.64,0.60,0.39,4.21,3.54,2.86,0.78,-0.56814,4.3,326.588,2.43,2.28,206000.0,6622382.0,22411.0,0.09
2026-02-23,3.64,3.64,0.60,0.34,4.21,3.54,2.86,0.78,-0.56814,4.3,326.588,2.40,2.26,206000.0,6622382.0,22411.0,0.09
2026-02-25,3.64,3.64,0.60,0.34,4.21,3.54,2.86,0.78,-0.56814,4.3,326.588,2.40,2.26,206000.0,6613395.0,22411.0,0.09


In [52]:
data_fed_transposed.loc['2007-01-01']

FEDFUNDS             5.25000
DFF                  5.17000
T10Y2Y              -0.11000
T10Y3M              -0.31000
GS10                 4.76000
GS2                  4.88000
BAMLH0A0HYM2         2.89000
BAMLC0A0CM           0.91000
NFCI                -0.62684
UNRATE               4.50000
CPIAUCSL           203.10000
T5YIE                2.26000
T10YIE               2.30000
ICSA            323000.00000
WALCL           865010.00000
M2SL              7080.10000
TEDRATE              0.47000
Name: 2007-01-01 00:00:00, dtype: float64

In [53]:
data_fed_transposed = data_fed_transposed.dropna(axis=0, how='any')
data_fed_transposed.head()

,FEDFUNDS,DFF,T10Y2Y,T10Y3M,GS10,GS2,BAMLH0A0HYM2,BAMLC0A0CM,NFCI,UNRATE,CPIAUCSL,T5YIE,T10YIE,ICSA,WALCL,M2SL,TEDRATE
2003-01-02,1.24,1.30,2.27,2.85,4.05,1.74,8.65,1.86,-0.46631,5.9,181.8,1.30,1.64,409000.0,732059.0,5779.4,0.18
2003-01-03,1.24,1.12,2.26,2.83,4.05,1.74,8.57,1.85,-0.47299,5.9,181.8,1.28,1.62,409000.0,732059.0,5779.4,0.19
2003-01-04,1.24,1.12,2.26,2.83,4.05,1.74,8.57,1.85,-0.47299,5.9,181.8,1.28,1.62,409000.0,732059.0,5779.4,0.19
2003-01-05,1.24,1.12,2.26,2.83,4.05,1.74,8.57,1.85,-0.47299,5.9,181.8,1.28,1.62,409000.0,732059.0,5779.4,0.19
2003-01-06,1.24,1.22,2.25,2.88,4.05,1.74,8.41,1.82,-0.47299,5.9,181.8,1.31,1.63,409000.0,732059.0,5779.4,0.20


In [54]:
data_fed_transposed.tail()

,FEDFUNDS,DFF,T10Y2Y,T10Y3M,GS10,GS2,BAMLH0A0HYM2,BAMLC0A0CM,NFCI,UNRATE,CPIAUCSL,T5YIE,T10YIE,ICSA,WALCL,M2SL,TEDRATE
2026-02-18,3.64,3.64,0.62,0.39,4.21,3.54,2.86,0.78,-0.56857,4.3,326.588,2.43,2.29,229000.0,6622382.0,22411.0,0.09
2026-02-19,3.64,3.64,0.61,0.39,4.21,3.54,2.88,0.79,-0.56857,4.3,326.588,2.43,2.29,206000.0,6622382.0,22411.0,0.09
2026-02-20,3.64,3.64,0.60,0.39,4.21,3.54,2.86,0.78,-0.56814,4.3,326.588,2.43,2.28,206000.0,6622382.0,22411.0,0.09
2026-02-23,3.64,3.64,0.60,0.34,4.21,3.54,2.86,0.78,-0.56814,4.3,326.588,2.40,2.26,206000.0,6622382.0,22411.0,0.09
2026-02-25,3.64,3.64,0.60,0.34,4.21,3.54,2.86,0.78,-0.56814,4.3,326.588,2.40,2.26,206000.0,6613395.0,22411.0,0.09


In [55]:
data_fed_transposed.to_parquet('data_fed.parquet')

## Transform events into one colum for event if not data and one colum of each type of data for those who have
#### saved as parquet

In [19]:
data_events.head()

,date,event,actual,previous
0,2026-01-28 00:00:00,US Federal Funds Rate,3.75%,3.75%
1,2025-12-10 00:00:00,US Federal Funds Rate,3.75%,4.00%
2,2025-10-29 00:00:00,US Federal Funds Rate,4.00%,4.25%
3,2025-09-17 00:00:00,US Federal Funds Rate,4.25%,4.50%
4,2025-07-30 00:00:00,US Federal Funds Rate,4.50%,4.50%


In [20]:
data_events.info()

<class 'pandas.DataFrame'>
RangeIndex: 4489 entries, 0 to 4488
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   date      4489 non-null   str  
 1   event     4489 non-null   str  
 2   actual    4489 non-null   str  
 3   previous  4489 non-null   str  
dtypes: str(4)
memory usage: 348.0 KB


In [21]:
cols = data_events['event'].unique()
cols

<ArrowStringArray>
[         'US Federal Funds Rate',                'US Core CPI m/m',
                     'US CPI m/m',                     'US CPI y/y',
                     'US PPI m/m',    'US Core PCE Price Index m/m',
  'US Non-Farm Employment Change',           'US Unemployment Rate',
 'US Average Hourly Earnings m/m',             'US Advance GDP q/q',
              'US Prelim GDP q/q',               'US Final GDP q/q',
            'US Retail Sales m/m',       'US Core Retail Sales m/m',
       'US Personal Spending m/m',         'US Personal Income m/m',
       'US ISM Manufacturing PMI',            'US ISM Services PMI',
            'US Building Permits',              'US Housing Starts',
         'US Existing Home Sales',              'US New Home Sales']
Length: 22, dtype: str

In [22]:
data_events.tail()

,date,event,actual,previous
4484,2007-05-24 00:00:00,US New Home Sales,981K,844K
4485,2007-04-25 00:00:00,US New Home Sales,858K,836K
4486,2007-03-26 00:00:00,US New Home Sales,848K,882K
4487,2007-02-28 00:00:00,US New Home Sales,937K,1123K
4488,2007-01-26 00:00:00,US New Home Sales,1120K,1069K


In [23]:
'''
in this transpose we don't need the previous value
we will fill forward the event day value which marks the previous value for the
dates forward
'''

#get all dates from original dataframe
data_events['date'] = pd.to_datetime(data_events['date'])

# Creates the transposed
data_events_transposed = data_events.pivot_table(
    index='date',
    columns='event',
    values='actual',
    aggfunc='first'
)

# fill forward
data_events_transposed = data_events_transposed.ffill(axis=0)

In [24]:
data_events_transposed.head()

event,US Advance GDP q/q,US Average Hourly Earnings m/m,US Building Permits,US CPI m/m,US CPI y/y,US Core CPI m/m,US Core PCE Price Index m/m,US Core Retail Sales m/m,US Existing Home Sales,US Federal Funds Rate,...,US ISM Manufacturing PMI,US ISM Services PMI,US New Home Sales,US Non-Farm Employment Change,US PPI m/m,US Personal Income m/m,US Personal Spending m/m,US Prelim GDP q/q,US Retail Sales m/m,US Unemployment Rate
date,,,,,,,,,,,,,,,,,,,,,
2007-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,51.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2007-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,51.4,57.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2007-01-05,NaN,0.5%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,51.4,57.1,NaN,167K,NaN,NaN,NaN,NaN,NaN,4.5%
2007-01-12,NaN,0.5%,NaN,NaN,NaN,NaN,NaN,1.0%,NaN,NaN,...,51.4,57.1,NaN,167K,NaN,NaN,NaN,NaN,0.9%,4.5%
2007-01-17,NaN,0.5%,NaN,NaN,NaN,NaN,NaN,1.0%,NaN,NaN,...,51.4,57.1,NaN,167K,0.9%,NaN,NaN,NaN,0.9%,4.5%


In [25]:
data_events_transposed = data_events_transposed.dropna(axis=0, how='any')
data_events_transposed.head()

event,US Advance GDP q/q,US Average Hourly Earnings m/m,US Building Permits,US CPI m/m,US CPI y/y,US Core CPI m/m,US Core PCE Price Index m/m,US Core Retail Sales m/m,US Existing Home Sales,US Federal Funds Rate,...,US ISM Manufacturing PMI,US ISM Services PMI,US New Home Sales,US Non-Farm Employment Change,US PPI m/m,US Personal Income m/m,US Personal Spending m/m,US Prelim GDP q/q,US Retail Sales m/m,US Unemployment Rate
date,,,,,,,,,,,,,,,,,,,,,
2007-03-29,3.5%,0.4%,1.53M,0.4%,2.4%,0.2%,0.3%,-0.1%,6.69M,5.25%,...,52.3,54.3,848K,97K,1.3%,1.0%,0.5%,2.2%,0.1%,4.5%
2007-03-30,3.5%,0.4%,1.53M,0.4%,2.4%,0.2%,0.3%,-0.1%,6.69M,5.25%,...,52.3,54.3,848K,97K,1.3%,0.6%,0.6%,2.2%,0.1%,4.5%
2007-04-02,3.5%,0.4%,1.53M,0.4%,2.4%,0.2%,0.3%,-0.1%,6.69M,5.25%,...,50.9,54.3,848K,97K,1.3%,0.6%,0.6%,2.2%,0.1%,4.5%
2007-04-04,3.5%,0.4%,1.53M,0.4%,2.4%,0.2%,0.3%,-0.1%,6.69M,5.25%,...,50.9,52.4,848K,97K,1.3%,0.6%,0.6%,2.2%,0.1%,4.5%
2007-04-06,3.5%,0.3%,1.53M,0.4%,2.4%,0.2%,0.3%,-0.1%,6.69M,5.25%,...,50.9,52.4,848K,180K,1.3%,0.6%,0.6%,2.2%,0.1%,4.4%


In [36]:
#need to clean the data and format it

def clean_values(val):
    if not isinstance(val, str):
        return val

    val = val.strip().replace(',', '').replace(' ', '')

    if val in ('', '-', 'N/A', 'nan'):
        return np.nan

    prefix = ''
    if val.startswith('<') or val.startswith('>'):
        prefix = val[0]
        val = val[1:]

    try:
        if val.endswith('%'):
            return float(val[:-1]) / 100
        elif val.upper().endswith('M'):
            return float(val[:-1]) * 1000000
        elif val.upper().endswith('K'):
            return float(val[:-1]) * 1000
        elif val.upper().endswith('B'):
            return float(val[:-1]) * 1000000000
        else:
            return float(val)
    except ValueError:
        return val


data_events_clean = data_events_transposed.apply(lambda col: col.map(clean_values))
data_events_clean.head()

event,US Advance GDP q/q,US Average Hourly Earnings m/m,US Building Permits,US CPI m/m,US CPI y/y,US Core CPI m/m,US Core PCE Price Index m/m,US Core Retail Sales m/m,US Existing Home Sales,US Federal Funds Rate,...,US ISM Manufacturing PMI,US ISM Services PMI,US New Home Sales,US Non-Farm Employment Change,US PPI m/m,US Personal Income m/m,US Personal Spending m/m,US Prelim GDP q/q,US Retail Sales m/m,US Unemployment Rate
date,,,,,,,,,,,,,,,,,,,,,
2007-03-29,0.035,0.004,1530000.0,0.004,0.024,0.002,0.003,-0.001,6690000.0,0.0525,...,52.3,54.3,848000.0,97000.0,0.013,0.010,0.005,0.022,0.001,0.045
2007-03-30,0.035,0.004,1530000.0,0.004,0.024,0.002,0.003,-0.001,6690000.0,0.0525,...,52.3,54.3,848000.0,97000.0,0.013,0.006,0.006,0.022,0.001,0.045
2007-04-02,0.035,0.004,1530000.0,0.004,0.024,0.002,0.003,-0.001,6690000.0,0.0525,...,50.9,54.3,848000.0,97000.0,0.013,0.006,0.006,0.022,0.001,0.045
2007-04-04,0.035,0.004,1530000.0,0.004,0.024,0.002,0.003,-0.001,6690000.0,0.0525,...,50.9,52.4,848000.0,97000.0,0.013,0.006,0.006,0.022,0.001,0.045
2007-04-06,0.035,0.003,1530000.0,0.004,0.024,0.002,0.003,-0.001,6690000.0,0.0525,...,50.9,52.4,848000.0,180000.0,0.013,0.006,0.006,0.022,0.001,0.044


In [37]:
data_events_clean.to_parquet('data_events.parquet')

### Events with only dates
#### saved as parquet

In [27]:
events_no_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 305 entries, 0 to 304
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   date      305 non-null    str  
 1   event     305 non-null    str  
 2   actual    305 non-null    str  
 3   previous  305 non-null    str  
dtypes: str(4)
memory usage: 24.0 KB


In [28]:
events_no_data['event'].unique()

<ArrowStringArray>
['US FOMC Statement', 'US FOMC Press Conference',
 'US FOMC Economic Projections']
Length: 3, dtype: str

In [29]:
events_no_data.tail()

,date,event,actual,previous
300,2012-04-25 00:00:00,US FOMC Economic Projections,None,None
301,2012-01-25 00:00:00,US FOMC Economic Projections,None,None
302,2011-11-02 00:00:00,US FOMC Economic Projections,None,None
303,2011-06-22 00:00:00,US FOMC Economic Projections,None,None
304,2011-04-27 00:00:00,US FOMC Economic Projections,None,None


In [30]:
'''
here we need to make a new col to calculate the next day of event
'''

#manually adds next event date, all of them are Mar 18 2026
future_event_dates = pd.to_datetime('2026-03-18')
cols = events_no_data['event'].unique()

last_event = pd.DataFrame({
    'date': [future_event_dates] * len(cols), #number of events
    'event': cols
})

#concat this event to the original df
events_no_data = pd.concat([events_no_data, last_event], ignore_index=True)

#format dates into datetime
events_no_data['date'] = pd.to_datetime(events_no_data['date'])

#creates a new col for the event day
events_no_data['event_date'] = events_no_data['date']
events_no_data['event_value'] = events_no_data['date']

#pivots one col for event
df_pivot = events_no_data.pivot_table(index='event_date',
                                columns='event',
                                values='event_value', #use a copy to avoid errors
                                aggfunc='first')

#creates a diary index for range
first_date = df_pivot.index.min()
last_date = df_pivot.index.max()
diary_index = pd.date_range(start=first_date, end=last_date, freq='D')

#reindex to include all days
df_diary = df_pivot.reindex(diary_index)

#backfill the dates
future_dates = df_diary.bfill()

#calculates the countdown for every date
df_countdown = future_dates.apply(lambda x: (x - future_dates.index).dt.days)

df_countdown.index.name = 'date'

df_countdown.tail()

event,US FOMC Economic Projections,US FOMC Press Conference,US FOMC Statement
date,,,
2026-03-14,4,4,4
2026-03-15,3,3,3
2026-03-16,2,2,2
2026-03-17,1,1,1
2026-03-18,0,0,0


In [31]:
df_countdown.to_parquet('event_day_countdown.parquet')
